# Imports

In [82]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder

# Dataset's preprocessing

In [83]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tunguz/200000-jeopardy-questions")

print("Path to dataset files:", path)

Path to dataset files: /Users/shpileva/.cache/kagglehub/datasets/tunguz/200000-jeopardy-questions/versions/1


In [84]:
DATASET_PATH = '/Users/shpileva/.cache/kagglehub/datasets/tunguz/200000-jeopardy-questions/versions/1/JEOPARDY_CSV.csv'

df = pd.read_csv(DATASET_PATH)

df = df.rename(columns={'Show Number' : 'show_number',
                        ' Air Date' : 'air_date',
                        ' Round' : 'round',
                        ' Category' : 'category',
                        ' Value' : 'value',
                        ' Question' : 'question',
                        ' Answer' : 'answer'})

df.head()

,show_number,air_date,round,category,value,question,answer
0,4680,2004-12-31,Jeopardy!,HISTORY,$200,"For the last 8 years of his life, Galileo was ...",Copernicus
1,4680,2004-12-31,Jeopardy!,ESPN's TOP 10 ALL-TIME ATHLETES,$200,No. 2: 1912 Olympian; football star at Carlisl...,Jim Thorpe
2,4680,2004-12-31,Jeopardy!,EVERYBODY TALKS ABOUT IT...,$200,The city of Yuma in this state has a record av...,Arizona
3,4680,2004-12-31,Jeopardy!,THE COMPANY LINE,$200,"In 1963, live on ""The Art Linkletter Show"", th...",McDonald's
4,4680,2004-12-31,Jeopardy!,EPITAPHS & TRIBUTES,$200,"Signer of the Dec. of Indep., framer of the Co...",John Adams


In [85]:
# балансируем датасет, чтобы убрать редко встречающиеся классы
min_examples = 50

category_counts = df["category"].value_counts()
valid_categories = category_counts[category_counts >= min_examples].index

df_filtered = df[df["category"].isin(valid_categories)].copy()

print("БАЛАНСИРОВКА")
print("Размер до фильтрации:", df.shape)
print("Размер после фильтрации:", df_filtered.shape)
print("Количество категорий:", df_filtered["category"].nunique())

#  выбираем топ N популярных категории
top_n = 5

top_categories = df_filtered["category"].value_counts().head(top_n).index
df_top = df_filtered[df_filtered["category"].isin(top_categories)].copy()

print(f"\nТОП {top_n} КАТЕГОРИЙ:")
print("Размер до фильтрации:", df_top.shape)
print("Размер после фильтрации:", df_top.shape)
print("Количество категорий:", df_top["category"].nunique())

БАЛАНСИРОВКА
Размер до фильтрации: (216930, 7)
Размер после фильтрации: (43201, 7)
Количество категорий: 348

ТОП 5 КАТЕГОРИЙ:
Размер до фильтрации: (2381, 7)
Размер после фильтрации: (2381, 7)
Количество категорий: 5


In [86]:
# сплитим
X = df_top["question"]
y = df_top["category"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape[0])
print("Test:", X_test.shape[0])

Train: 1904
Test: 477


# Model's Pipeline 

### Baseline

In [87]:
from sklearn.svm import LinearSVC

model_svc = Pipeline([
    ("features", FeatureUnion([
        ("word", TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            max_features=100_000,
            ngram_range=(1, 3),
            min_df=2,
            sublinear_tf=True
        )),
        ("char", TfidfVectorizer(
            analyzer="char_wb",
            lowercase=True,
            max_features=50_000,
            ngram_range=(3, 5),
            min_df=2,
            sublinear_tf=True
        ))
    ])),
    ("clf", LinearSVC(
        C=1.0,
        class_weight="balanced"
    ))
])

model_svc.fit(X_train, y_train)

y_pred = model_svc.predict(X_test)

print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print("Weighted F1:", f1_score(y_test, y_pred, average="weighted"))
print(classification_report(y_test, y_pred))

Macro F1: 0.7409070797755544
Weighted F1: 0.753368587035673
                  precision    recall  f1-score   support

AMERICAN HISTORY       0.78      0.81      0.80        84
  BEFORE & AFTER       0.85      0.84      0.84       110
      LITERATURE       0.83      0.88      0.85        99
       POTPOURRI       0.48      0.42      0.45        80
         SCIENCE       0.75      0.77      0.76       104

        accuracy                           0.76       477
       macro avg       0.74      0.74      0.74       477
    weighted avg       0.75      0.76      0.75       477

